# 🔤 Brahmi Script OCR — TrOCR Training on Google Colab

This notebook trains TrOCR on a **map.json-driven Brahmi dataset** using Colab GPU.

**Pipeline:**
1. Mount Google Drive & clone project
2. Install dependencies
3. Load curated dataset (optional ZIP from Drive)
4. Validate dataset (`dataset/validate_dataset.py`)
5. Fine-tune `microsoft/trocr-small-printed`
6. Test inference + debug JSON trace
7. Save model to Drive

---

> ⚠️ **Before running:** Go to **Runtime → Change runtime type → GPU (T4)**


## Step 0 — Verify GPU is available

In [20]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → GPU")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU device      : Tesla T4
GPU memory      : 15.6 GB


## Step 1 — Mount Google Drive

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
%cd /content


/content


## Step 2 — Clone project from GitHub

In [33]:
import os
import shutil

PROJECT_DIR = '/content/brahmi_ocr_project'

# Clone fresh from GitHub
if os.path.exists(PROJECT_DIR):
    !rm -rf {PROJECT_DIR}

!git clone https://github.com/tejaskarade100/brahmi_ocr_project.git {PROJECT_DIR}

%cd {PROJECT_DIR}
!ls

Cloning into '/content/brahmi_ocr_project'...
remote: Enumerating objects: 345, done.
remote: Counting objects: 100% (345/345), done.
remote: Compressing objects: 100% (242/242), done.
remote: Total 345 (delta 130), reused 297 (delta 85), pack-reused 0 (from 0)
Receiving objects: 100% (345/345), 557.69 KiB | 3.28 MiB/s, done.
Resolving deltas: 100% (130/130), done.
/content/brahmi_ocr_project
1.jpeg			inference		    test_tokenizer.py
backend			kaggle_training.ipynb	    training
brahmi.json		new.txt			    utils
colab_training.ipynb	NotoSansBrahmi-Regular.ttf  webapp
dataset			README.md
implementation_plan.md	requirements.txt


## Step 3 — Install dependencies

In [34]:
!pip install -q -r requirements.txt

## Step 4a — Load curated map.json dataset

If a curated dataset ZIP exists on Drive, unzip it into the project. Otherwise use the repository dataset as-is.


In [35]:
import os
import shutil

PROJECT_DIR = '/content/brahmi_ocr_project'
DATASET_DIR = os.path.join(PROJECT_DIR, 'dataset')
LOCAL_DATASET_ZIP = '/content/curated_dataset.zip' # Moved out of project dir for safety
DRIVE_DATASET_ZIP_CANDIDATES = [
    '/content/drive/MyDrive/Project/brahmi_ocr_project/curated_dataset.zip',
    '/content/drive/MyDrive/Project/brahmi_ocr_project/dataset.zip',
    '/content/drive/MyDrive/curated_dataset.zip',
    '/content/drive/MyDrive/dataset.zip',
]

zip_path = next((p for p in DRIVE_DATASET_ZIP_CANDIDATES if os.path.exists(p)), None)

if zip_path:
    print(f'📦 Found curated dataset ZIP: {zip_path}')

    # Copy zip to safe root location first
    shutil.copy2(zip_path, LOCAL_DATASET_ZIP)

    # Wipe old dataset dir if it exists to prevent corruption
    if os.path.exists(DATASET_DIR):
        shutil.rmtree(DATASET_DIR)

    # Recreate the empty dataset dir
    os.makedirs(DATASET_DIR, exist_ok=True)

    # Unzip directly INTO the dataset dir
    !unzip -q -o {LOCAL_DATASET_ZIP} -d {DATASET_DIR}
    print('✅ Dataset loaded from Drive ZIP.')
else:
    print('ℹ️ No dataset ZIP found on Drive; using repository dataset folder.')

if not os.path.exists(os.path.join(DATASET_DIR, 'map.json')):
    raise FileNotFoundError('dataset/map.json not found after dataset setup. Double check your ZIP structure!')

print('✅ map.json detected.')


📦 Found curated dataset ZIP: /content/drive/MyDrive/Project/brahmi_ocr_project/curated_dataset.zip
✅ Dataset loaded from Drive ZIP.
✅ map.json detected.


In [36]:
# Validate dataset and generate JSON report
!python dataset/validate_dataset.py --data_dir dataset --json_out dataset_validation_report.json

import json
with open('dataset_validation_report.json', 'r', encoding='utf-8') as f:
    report = json.load(f)

print('Total samples            :', report['total_image_samples'])
print('Mapped classes           :', report['mapped_class_count'])
print('Mapped classes w/ images :', report['mapped_classes_with_images'])
print('Warnings                 :', len(report.get('warnings', [])))
print('Errors                   :', len(report.get('errors', [])))



=== Dataset Validation Report ===
Data dir                 : /content/brahmi_ocr_project/dataset
Mapped classes           : 597
Mapped with images       : 393
Mapped missing folder    : 204
Mapped empty             : 0
Total samples            : 67200
Composition              : Chars/NGrams=58.33% | Words=14.88% | Sentences+Phrases=26.79%

Target ratio check (with tolerance):
  characters_ngrams    OK  | target=0.600 actual=0.583 diff=0.017
  words                OUT | target=0.250 actual=0.149 diff=0.101
  sentences_phrases    OUT | target=0.150 actual=0.268 diff=0.118

Warnings (2):
  - Mapped folders missing on disk: 204
  - Dataset composition is outside target tolerance

JSON report written to   : /content/brahmi_ocr_project/dataset_validation_report.json
Total samples            : 67200
Mapped classes           : 597
Mapped classes w/ images : 393
Warnings                 : 2
Errors                   : 0


## Step 4b — Quick class coverage preview


In [37]:
import json
with open('dataset_validation_report.json', 'r', encoding='utf-8') as f:
    report = json.load(f)

print('Top classes by image count:')
for folder, count in report.get('top_classes_by_count', [])[:10]:
    print(f'  {count:6d}  {folder}')

print('\nBottom classes by image count:')
for folder, count in report.get('bottom_classes_by_count', [])[:10]:
    print(f'  {count:6d}  {folder}')


Top classes by image count:
   28000  5Words_Phrases
     100  1Vowels/1_a
     100  1Vowels/3_i
     100  1Vowels/5_u
     100  1Vowels/11_e
     100  1Vowels/12_ai
     100  1Vowels/13_o
     100  1Vowels/14_au
     100  2Consonants/1_Ka_group/ka/1_base
     100  2Consonants/1_Ka_group/ka/2_aa

Bottom classes by image count:
       0  1Vowels/2_ā
       0  1Vowels/4_ī
       0  1Vowels/6_ū
       0  1Vowels/7_ṛ
       0  1Vowels/8_ṝ
       0  1Vowels/9_l̥
       0  1Vowels/10_l̥̄
       0  2Consonants/1_Ka_group/ṅa/1_base
       0  2Consonants/1_Ka_group/ṅa/2_aa
       0  2Consonants/1_Ka_group/ṅa/3_i


## Step 4c — Optional strict validation gate


In [38]:
STRICT_VALIDATE = False  # set True to fail fast on warnings/ratio mismatch

if STRICT_VALIDATE:
    !python dataset/validate_dataset.py --data_dir dataset --strict
else:
    print('Skipping strict gate (STRICT_VALIDATE=False)')


Skipping strict gate (STRICT_VALIDATE=False)


## Step 5 — Train the TrOCR model 🚀

| Setting | Value |
|---------|-------|
| Base model | `microsoft/trocr-small-printed` |
| Epochs | 15 |
| Batch size | 8 |
| Learning rate | 3e-5 |
| FP16 | Auto (enabled on GPU) |
| Data source | `dataset/map.json` + mapped folders |

**Estimated time:** depends on dataset size and GPU tier.


In [ ]:
!python training/train.py \
    --epochs 15 \
    --batch_size 8 \
    --gradient_accumulation_steps 4 \
    --balanced_sampling \
    --lr 3e-5 \
    --warmup_ratio 0.1 \
    --patience 3 \
    --data_dir dataset \
    --output_dir model/brahmi_trocr \
    --drive_save_path /content/drive/MyDrive/Project/brahmi_ocr_project/model/brahmi_trocr \
    --train_ratio 0.8 \
    --val_ratio 0.1 \
    --test_ratio 0.1 \
    --image_size 384


Device: cuda | FP16: True
preprocessor_config.json: 100% 272/272 [00:00<00:00, 1.34MB/s]
config.json: 4.21kB [00:00, 11.6MB/s]
tokenizer_config.json: 100% 327/327 [00:00<00:00, 1.89MB/s]
special_tokens_map.json: 100% 238/238 [00:00<00:00, 1.03MB/s]
sentencepiece.bpe.model: 100% 1.36M/1.36M [00:01<00:00, 954kB/s] 

Train Dataset Breakdown
  Total samples         : 47488
  Characters/N-Grams    : 25172 (53.01%)
  Words                 : 7987 (16.82%)
  Phrases               : 7952 (16.75%)
  Long sentences        : 6377 (13.43%)

Val Dataset Breakdown
  Total samples         : 5936
  Characters/N-Grams    : 3074 (51.79%)
  Words                 : 998 (16.81%)
  Phrases               : 1036 (17.45%)
  Long sentences        : 828 (13.95%)
model.safetensors: 100% 246M/246M [00:02<00:00, 94.5MB/s]
Loading weights: 100% 360/360 [00:00<00:00, 787.40it/s, Materializing param=encoder.layernorm.weight]
VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-printed
Key                  

## Step 6 — Test inference on sample images

In [ ]:
# Run inference on first test sample from map-driven dataset
import sys
sys.path.insert(0, PROJECT_DIR)

from transformers import TrOCRProcessor
from training.dataset_loader import BrahmiDataset

processor_for_ds = TrOCRProcessor.from_pretrained('model/brahmi_trocr', use_fast=False)
test_ds = BrahmiDataset('dataset', split='test', processor=processor_for_ds, seed=42)

if len(test_ds.samples) == 0:
    print('No test samples available.')
else:
    sample = test_ds.samples[0]
    print(f'Test image   : {sample.image_path}')
    print(f'Ground truth : {sample.label_text}')
    !python inference/predict.py --image {sample.image_path} --model_dir model/brahmi_trocr --preprocess --threshold_method auto --multiline --debug


In [ ]:
# Batch test: compare predictions vs ground truth (map-driven test split)
from inference.predict import load_trained_model, predict
from transformers import TrOCRProcessor
from training.dataset_loader import BrahmiDataset

if os.path.exists('model/brahmi_trocr'):
    ds_processor = TrOCRProcessor.from_pretrained('model/brahmi_trocr', use_fast=False)
    test_ds = BrahmiDataset('dataset', split='test', processor=ds_processor, seed=42)

    processor, model, device = load_trained_model('model/brahmi_trocr')
    print(f'Model loaded on: {device}\n')

    correct = 0
    total = min(10, len(test_ds.samples))
    for sample in test_ds.samples[:total]:
        result = predict(sample.image_path, processor, model, device, preprocess=True, multiline=True, debug=False)
        pred = result['predicted_text']
        gt = sample.label_text
        match = '✅' if pred.strip() == gt.strip() else '❌'
        if pred.strip() == gt.strip():
            correct += 1
        print(f'{match} {os.path.basename(sample.image_path)}  |  GT: {gt}  |  Pred: {pred}')

    if total > 0:
        print(f'\nAccuracy: {correct}/{total} ({100*correct/total:.0f}%)')
else:
    print('Trained model not found at model/brahmi_trocr')


## Step 7 — Save trained model to Google Drive

Copies the model to your Drive at: `My Drive/Project/brahmi_ocr_project/model/brahmi_trocr/`

In [ ]:
import shutil

# Save model to your Google Drive project folder
DRIVE_MODEL_DIR = '/content/drive/MyDrive/Project/brahmi_ocr_project/model/brahmi_trocr'

os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)

if os.path.exists(DRIVE_MODEL_DIR):
    shutil.rmtree(DRIVE_MODEL_DIR)

shutil.copytree('model/brahmi_trocr', DRIVE_MODEL_DIR)
print(f"✅ Model saved to Google Drive!")
print(f"   Path: {DRIVE_MODEL_DIR}")
print(f"   Files: {os.listdir(DRIVE_MODEL_DIR)}")

## Done! 🎉

### Your trained model is saved at:
📁 `Google Drive / Project / brahmi_ocr_project / model / brahmi_trocr /`

### To use locally:
1. Download the `brahmi_trocr` folder from Drive to your local `model/` folder
2. Run inference:
   ```bash
   python inference/predict.py --image <your_image.png> --model_dir model/brahmi_trocr
   ```